<a href="https://colab.research.google.com/github/emordecai/SDM_Aedes_sierrensis/blob/main/Aedes_sierrensis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import ee
import geemap

# Trigger the authentication flow.
ee.Authenticate()

# Initialize the library. Using a project you have access to, or omitting for default.
#ee.Initialize()
ee.Initialize(project='gbsc-gcp-lab-emordeca') #(original set up working in the lab cloud project)

In [ ]:
########################
# 1. Load Spatial Data
########################

# Load point locations (Aedes sierrensis occurrences and background points [thinned])
occ_bg = ee.FeatureCollection('projects/gbsc-gcp-lab-emordeca/assets/aedes-sierrensis/aedes_sierrensis_all_spp_thinned_since1981') # fill in this directory once the file is uploaded to assets

# Buffer each point to make a 1 km radius (with max error 100m)
point_buffers = occ_bg.map(lambda f: f.buffer(1000, 100))

# create point subsets for Aedes sierrensis vs. background points
target_species = 'Aedes sierrensis'
occ = occ_bg.filter(ee.Filter.eq('scientificName', target_species))
bg = occ_bg.filter(ee.Filter.neq('scientificName', target_species))


# # Load spatial extent of interest (USA, Canada, and Mexico)
countries = ee.FeatureCollection('FAO/GAUL/2015/level0')
na_countries = countries.filter(ee.Filter.inList('ADM0_NAME', ['United States', 'Canada', 'Mexico', 'Guatemala', 'Nicaragua', 'Costa Rica', 'Panama', 'Cuba', 'Puerto Rico', 'Bahamas', 'El Salvador', 'Belize']))
na_boundary = na_countries.geometry()


In [ ]:
### Map it!
Map = geemap.Map(center= [45, -115], zoom = 3) # arbitrarily centering the map on Salt Lake City and zooming to relevant area
Map.addLayer(point_buffers, {'color': 'gray'}, 'point_buffers')
Map.addLayer(occ, {'color': 'turquoise'}, 'occ') # create a subset that is just Aedes sierrensis and map it
Map.addLayer(bg, {'color': 'red'}, 'bg') # create another subset that is just background points and map it
Map

Map(center=[45, -115], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright', tr…

In [ ]:
### Bioclim variables
### Selecting all bioclim variables (to subset later)

dataset = ee.Image('WORLDCLIM/V1/BIO')

# extract data: reduce regions to get mean value within buffer
# scale is ~1km
bioclim_stats = dataset.reduceRegions(
    collection=point_buffers,
    reducer=ee.Reducer.mean(),
    scale=1000,
    crs='EPSG:4326'
)

# Optional: print first row to check
print(bioclim_stats.first().getInfo())

# Export to Google Drive
ee.batch.Export.table.toDrive(
    collection=bioclim_stats,
    description='aedes_bioclim_mean_2026-06-24',
    folder='SDM_learning_project',
    fileFormat='CSV').start()

{'type': 'Feature', 'geometry': {'type': 'Polygon', 'coordinates': [[[-72.27355132525508, 42.49094824325248], [-72.28079131182186, 42.489193236576696], [-72.28520685023194, 42.48461290110645], [-72.2850762288976, 42.47899393684606], [-72.28045144933387, 42.474527726447704], [-72.27313645435352, 42.47295571500594], [-72.26598319833832, 42.47489077713751], [-72.26178064288264, 42.4795784912096], [-72.26216819946032, 42.48519100827366], [-72.2669957753836, 42.48953940698673], [-72.27355132525508, 42.49094824325248]]]}, 'id': '00000000000000000114', 'properties': {'bio01': 75.2670520231214, 'bio02': 126.60809248554914, 'bio03': 31.218497109826593, 'bio04': 9317.15260115607, 'bio05': 273.29942196531795, 'bio06': -123.18381502890176, 'bio07': 396.4832369942197, 'bio08': 170.07283236994223, 'bio09': -37.06127167630058, 'bio10': 191.67283236994223, 'bio11': -50.73294797687862, 'bio12': 1132.3653179190753, 'bio13': 102.39190751445086, 'bio14': 79.67745664739884, 'bio15': 7.6000000000000005, 'bi

In [ ]:
# ########################## OLD VERSION: Not used for this SDM analysis
# #########################
# ### 2. Climate data
# ### a. Temperature - ERA5-Land Monthly
# #########################

# temp = ee.ImageCollection('ECMWF/ERA5_LAND/MONTHLY_AGGR') \
#     .select('temperature_2m') \
#     .toBands() \
#     .select(ee.List.sequence(599, 874))


# # Clip to countries of interest
# temp = temp.clip(na_countries) # clipping to North American Countries

# # Reduce to spatial units (monthly mean temperature)
# temp_stats = temp.reduceRegions(
#     collection=point_buffers,
#     reducer=ee.Reducer.mean(),
#     scale=11132)

# # Export
# ee.batch.Export.table.toDrive(
#     collection=temp_stats,
#     description='aedes_temperature_monthly_mean',
#     folder='SDM_learning_project',
#     fileFormat='CSV').start()



In [ ]:
# ########################## OLD VERSION: Not used for this SDM analysis
# #########################
# ### 2. Climate data
# ### b. Precipitation
# #########################

# precip = ee.ImageCollection('ECMWF/ERA5_LAND/MONTHLY_AGGR') \
#     .select('total_precipitation_sum') \
#     .toBands() \
#     .select(ee.List.sequence(599, 874))

# # Clip to countries of interest
# precip = precip.clip(na_countries)

# # Reduce to spatial units (monthly sum of precip)
# precip_stats = precip.reduceRegions(
#     collection=point_buffers,
#     reducer=ee.Reducer.sum(),
#     scale=11132)

# # Export
# ee.batch.Export.table.toDrive(
#     collection=precip_stats,
#     description='aedes_precipitation_monthly_sum',
#     folder='SDM_learning_project',
#     fileFormat='CSV').start()


In [ ]:
#########################
### 3. Land cover data
#########################

# 30 m resolution land cover data for USA, Mexico, and Canada collected from 2019-2021 across different parts of the region
# From Landsat 8 Collection 2 Level 1, classified and harmonized
# Canada images from 2020, coterminous US from 2019, Alaska from 2021, Mexico from land cover change from 2015-2020
dataset = ee.Image("USGS/NLCD_RELEASES/2020_REL/NALCMS")
landCover = dataset.select('landcover')

# extract average land cover value using reduceRegions
landcover_stats = landCover.reduceRegions(
    collection=point_buffers,
    reducer=ee.Reducer.mode(maxRaw=1e5),  # using mode to classify the majority land cover type (since the categories aren't meaningful as an average)
    scale=30,
    crs='EPSG:5070'
)

# Optional: print first result to verify
print(landcover_stats.first().getInfo())

# Export to Google Drive
ee.batch.Export.table.toDrive(
    collection=landcover_stats,
    description='aedes_landcover_mode_2026-06-24',
    folder='SDM_learning_project',
    fileFormat='CSV').start()


{'type': 'Feature', 'geometry': {'type': 'Polygon', 'coordinates': [[[-72.27355132525508, 42.49094824325248], [-72.28079131182186, 42.489193236576696], [-72.28520685023194, 42.48461290110645], [-72.2850762288976, 42.47899393684606], [-72.28045144933387, 42.474527726447704], [-72.27313645435352, 42.47295571500594], [-72.26598319833832, 42.47489077713751], [-72.26178064288264, 42.4795784912096], [-72.26216819946032, 42.48519100827366], [-72.2669957753836, 42.48953940698673], [-72.27355132525508, 42.49094824325248]]]}, 'id': '00000000000000000114', 'properties': {'mode': 5, 'presence': 0, 'row_code': 277, 'scientificName': 'Aedes communis', 'year': 2018.42105263158}}


In [ ]:
# ### Bioclim variables, v2 of the SDM
# ### Selects a new set of bioclim variables to focus on for SDMs
# # bio06 = min temp coldest month
# # bio08 = mean temp wettest quarter
# # bio11 = mean temp coldest quarter
# # bio04 = temperature seasonality
# # bio16 = precip of wettest quarter
# # bio15 = precip seasonality

# dataset = ee.Image('WORLDCLIM/V1/BIO')
# selected_bands = dataset.select(['bio06', 'bio08', 'bio11', 'bio04', 'bio16', 'bio15'])

# # extract data: reduce regions to get mean value within buffer
# # scale is ~1km
# bioclim_stats = selected_bands.reduceRegions(
#     collection=point_buffers,
#     reducer=ee.Reducer.mean(),
#     scale=1000,
#     crs='EPSG:4326'
# )

# # Optional: print first row to check
# print(bioclim_stats.first().getInfo())

# # Export to Google Drive
# ee.batch.Export.table.toDrive(
#     collection=bioclim_stats,
#     description='aedes_bioclim_mean_2026-04-03',
#     folder='SDM_learning_project',
#     fileFormat='CSV').start()

{'type': 'Feature', 'geometry': {'type': 'Polygon', 'coordinates': [[[-118.75148843879322, 36.5714425446915], [-118.75813590792849, 36.569687581330015], [-118.76219022951837, 36.565107314805594], [-118.76207047718613, 36.55948834804503], [-118.75782414197703, 36.55502206723167], [-118.75110750195577, 36.553450016587185], [-118.74453938217194, 36.55538512590815], [-118.74068074829866, 36.56007290713973], [-118.74103677606007, 36.56568541678673], [-118.7454693986771, 36.570033743937614], [-118.75148843879322, 36.5714425446915]]]}, 'id': '00000000000000000129', 'properties': {'bio04': 5911.600257069409, 'bio06': -42.699228791773784, 'bio08': 13.475578406169667, 'bio11': 12.42030848329049, 'bio15': 78.88046272493573, 'bio16': 386.3920308483291, 'presence': 1, 'row_code': 298, 'scientificName': 'Aedes sierrensis', 'year': 1923}}


In [ ]:
# #### Original Bioclim variables used in v1 of the SDM analysis


# ### Bioclim long-term climatology variables: https://developers.google.com/earth-engine/datasets/catalog/WORLDCLIM_V1_BIO#bands
# # Bioclim variables of interest:
# # bio01 = annual mean temperature
# # bio03 = isothermality (100* mean diurnal range / annual range)
# # bio04 = temperature seasonality (standard deviation * 100)
# # bio05 = max temperature of warmest month
# # bio06 = min temperature of coldest month
# # bio08 = mean temperature of wettest quarter
# # bio09 = mean temperature of driest quarter
# # bio12 = annual precipitation
# # bio15 = precipitation seasonality

# dataset = ee.Image('WORLDCLIM/V1/BIO')
# selected_bands = dataset.select(['bio01', 'bio03', 'bio04', 'bio05', 'bio06', 'bio08', 'bio09', 'bio12', 'bio15'])

# # extract data: reduce regions to get mean value within buffer
# # scale is ~1km
# bioclim_stats = selected_bands.reduceRegions(
#     collection=point_buffers,
#     reducer=ee.Reducer.mean(),
#     scale=1000,
#     crs='EPSG:4326'
# )

# # Optional: print first row to check
# print(bioclim_stats.first().getInfo())

# # Export to Google Drive
# ee.batch.Export.table.toDrive(
#     collection=bioclim_stats,
#     description='aedes_bioclim_mean_2026-003-09',
#     folder='SDM_learning_project',
#     fileFormat='CSV').start()


{'type': 'Feature', 'geometry': {'type': 'Polygon', 'coordinates': [[[-118.75148843879322, 36.5714425446915], [-118.75813590792849, 36.569687581330015], [-118.76219022951837, 36.565107314805594], [-118.76207047718613, 36.55948834804503], [-118.75782414197703, 36.55502206723167], [-118.75110750195577, 36.553450016587185], [-118.74453938217194, 36.55538512590815], [-118.74068074829866, 36.56007290713973], [-118.74103677606007, 36.56568541678673], [-118.7454693986771, 36.570033743937614], [-118.75148843879322, 36.5714425446915]]]}, 'id': '00000000000000000129', 'properties': {'bio01': 76.11053984575837, 'bio03': 42, 'bio04': 5911.600257069409, 'bio05': 242.96143958868896, 'bio06': -42.699228791773784, 'bio08': 13.475578406169667, 'bio09': 156.0694087403599, 'bio12': 762.280205655527, 'bio15': 78.88046272493573, 'presence': 1, 'row_code': 298, 'scientificName': 'Aedes sierrensis', 'year': 1923}}


In [ ]:
### Extract bioclim data for the full geographic extent

# --------------------------------
# Load WorldClim bioclim dataset
# --------------------------------
dataset = ee.Image("WORLDCLIM/V1/BIO")

# selected_bands = dataset.select([
#     "bio01",
#     "bio03",
#     "bio04",
#     "bio05",
#     "bio06",
#     "bio08",
#     "bio09",
#     "bio12",
#     "bio15"
# ])

# --------------------------------
# Define North America region
# --------------------------------
countries = ee.FeatureCollection("USDOS/LSIB_SIMPLE/2017")

north_america = countries.filter(
    ee.Filter.inList("country_na", ["United States", "Canada", "Mexico"])
)

# Use bounding box to avoid complex geometry
region = north_america.geometry().bounds()

# --------------------------------
# Reproject to 1 km
# --------------------------------
bioclim_na = dataset.reproject(
    crs="EPSG:4326",
    scale=1000
)

# --------------------------------
# Export to Google Drive
# --------------------------------
task = ee.batch.Export.image.toDrive(
    image=bioclim_na,
    description="worldclim_bioclim_NA",
    folder="SDM_learning_project",
    fileNamePrefix="worldclim_bioclim_na",
    region=region,
    scale=1000,
    crs="EPSG:4326",
    maxPixels=1e13
)

task.start()

print("Export started")
print(task.status())


NameError: name 'ee' is not defined

In [ ]:
# #############################
# ### NALCMS land cover dataset: https://developers.google.com/earth-engine/datasets/catalog/USGS_NLCD_RELEASES_2020_REL_NALCMS#code-editor-javascript
# ### Extract land cover data for all of US, CA, MX


# # ---------------------------
# # Load landcover
# # ---------------------------
# dataset = ee.Image("USGS/NLCD_RELEASES/2020_REL/NALCMS")
# landcover = dataset.select("landcover")

# # ---------------------------
# # Define North America bounding box
# # ---------------------------
# countries = ee.FeatureCollection("USDOS/LSIB_SIMPLE/2017")
# north_america = countries.filter(
#     ee.Filter.inList("country_na", ["United States", "Canada", "Mexico"])
# )
# na_bbox = north_america.geometry().bounds()  # simple rectangle

# # ---------------------------
# # Generate a grid of 10°x10° tiles over North America
# # ---------------------------
# def create_grid(bounds, dx=10, dy=10):
#     coords = bounds.coordinates().getInfo()[0]
#     min_x = min([c[0] for c in coords])
#     max_x = max([c[0] for c in coords])
#     min_y = min([c[1] for c in coords])
#     max_y = max([c[1] for c in coords])

#     tiles = []
#     x = min_x
#     while x < max_x:
#         y = min_y
#         while y < max_y:
#             tile = ee.Geometry.Rectangle([x, y, min(x+dx, max_x), min(y+dy, max_y)])
#             tiles.append(tile)
#             y += dy
#         x += dx
#     return tiles

# tiles = create_grid(na_bbox, dx=10, dy=10)

# # ---------------------------
# # Loop over tiles and submit exports
# # ---------------------------
# tasks = []

# for i, tile in enumerate(tiles):
#     # Downscale 30m to 1km majority
#     lc_1km = landcover.reduceResolution(
#         reducer=ee.Reducer.mode(),
#         maxPixels=1024
#     ).reproject(
#         crs="EPSG:4326",
#         scale=1000
#     ).clip(tile)

#     # Binary bands
#     forest = lc_1km.remap([1,2,3,4,5,6],[1,1,1,1,1,1],0).rename("forest")
#     shrub = lc_1km.remap([7,8],[1,1],0).rename("shrubland")
#     grass = lc_1km.remap([9,10],[1,1],0).rename("grassland")
#     wetland = lc_1km.remap([14],[1],0).rename("wetland")
#     crop = lc_1km.remap([15],[1],0).rename("cropland")
#     urban = lc_1km.remap([17],[1],0).rename("urban")
#     water = lc_1km.remap([18],[1],0).rename("water")

#     lc_predictors = ee.Image.cat([forest, shrub, grass, wetland, crop, urban, water])

#     # Export task
#     task = ee.batch.Export.image.toDrive(
#         image=lc_predictors,
#         description=f"lc_tile_{i}",
#         folder="SDM_learning_project",
#         fileNamePrefix=f"lc_tile_{i}",
#         region=tile,
#         scale=1000,
#         crs="EPSG:4326",
#         maxPixels=1e13
#     )
#     task.start()
#     tasks.append(task)

# print(f"Submitted {len(tasks)} tile exports.")

Submitted 63 tile exports.


In [ ]:
# import time

# # tasks = list of your submitted tile tasks
# # e.g., tasks = [task0, task1, task2, ...]
# while any(t.active() for t in tasks):
#     for t in tasks:
#         status = t.status()
#         print(f"{status['description']}: {status['state']}")
#     print("Waiting 30 seconds...\n")
#     time.sleep(30)

# # Print final status for all tasks
# print("All tasks completed (or failed):")
# for t in tasks:
#     status = t.status()
#     state = status['state']
#     desc = status['description']
#     print(f"{desc} final state: {state}")
#     if state == 'FAILED':
#         print(f"  Error message: {status.get('error_message')}")

lc_tile_0: READY
lc_tile_1: READY
lc_tile_2: READY
lc_tile_3: READY
lc_tile_4: READY
lc_tile_5: READY
lc_tile_6: READY
lc_tile_7: READY
lc_tile_8: READY
lc_tile_9: READY
lc_tile_10: READY
lc_tile_11: READY
lc_tile_12: READY
lc_tile_13: READY
lc_tile_14: READY
lc_tile_15: READY
lc_tile_16: READY
lc_tile_17: READY
lc_tile_18: READY
lc_tile_19: READY
lc_tile_20: READY
lc_tile_21: READY
lc_tile_22: READY
lc_tile_23: READY
lc_tile_24: READY
lc_tile_25: READY
lc_tile_26: READY
lc_tile_27: READY
lc_tile_28: READY
lc_tile_29: READY
lc_tile_30: READY
lc_tile_31: READY
lc_tile_32: READY
lc_tile_33: READY
lc_tile_34: READY
lc_tile_35: READY
lc_tile_36: READY
lc_tile_37: READY
lc_tile_38: READY
lc_tile_39: READY
lc_tile_40: READY
lc_tile_41: READY
lc_tile_42: READY
lc_tile_43: READY
lc_tile_44: READY
lc_tile_45: READY
lc_tile_46: READY
lc_tile_47: READY
lc_tile_48: READY
lc_tile_49: READY
lc_tile_50: READY
lc_tile_51: READY
lc_tile_52: READY
lc_tile_53: READY
lc_tile_54: READY
lc_tile_55: READY
lc

Streaming output truncated to the last 5000 lines.
lc_tile_5: READY
lc_tile_6: READY
lc_tile_7: READY
lc_tile_8: READY
lc_tile_9: READY
lc_tile_10: READY
lc_tile_11: READY
lc_tile_12: READY
lc_tile_13: READY
lc_tile_14: READY
lc_tile_15: READY
lc_tile_16: READY
lc_tile_17: READY
lc_tile_18: READY
lc_tile_19: READY
lc_tile_20: READY
lc_tile_21: READY
lc_tile_22: READY
lc_tile_23: READY
lc_tile_24: READY
lc_tile_25: READY
lc_tile_26: READY
lc_tile_27: READY
lc_tile_28: READY
lc_tile_29: READY
lc_tile_30: READY
lc_tile_31: READY
lc_tile_32: READY
lc_tile_33: READY
lc_tile_34: READY
lc_tile_35: READY
lc_tile_36: READY
lc_tile_37: READY
lc_tile_38: READY
lc_tile_39: READY
lc_tile_40: READY
lc_tile_41: READY
lc_tile_42: READY
lc_tile_43: READY
lc_tile_44: READY
lc_tile_45: READY
lc_tile_46: READY
lc_tile_47: READY
lc_tile_48: CANCELLED
lc_tile_49: CANCELLED
lc_tile_50: CANCELLED
lc_tile_51: CANCELLED
lc_tile_52: CANCELLED
lc_tile_53: CANCELLED
lc_tile_54: CANCELLED
lc_tile_55: CANCELLED
lc_t

Streaming output truncated to the last 5000 lines.
lc_tile_5: READY
lc_tile_6: READY
lc_tile_7: READY
lc_tile_8: READY
lc_tile_9: READY
lc_tile_10: READY
lc_tile_11: READY
lc_tile_12: READY
lc_tile_13: READY
lc_tile_14: READY
lc_tile_15: READY
lc_tile_16: READY
lc_tile_17: READY
lc_tile_18: READY
lc_tile_19: READY
lc_tile_20: READY
lc_tile_21: READY
lc_tile_22: READY
lc_tile_23: READY
lc_tile_24: READY
lc_tile_25: READY
lc_tile_26: READY
lc_tile_27: READY
lc_tile_28: READY
lc_tile_29: READY
lc_tile_30: READY
lc_tile_31: READY
lc_tile_32: READY
lc_tile_33: READY
lc_tile_34: READY
lc_tile_35: READY
lc_tile_36: READY
lc_tile_37: READY
lc_tile_38: READY
lc_tile_39: READY
lc_tile_40: READY
lc_tile_41: READY
lc_tile_42: READY
lc_tile_43: READY
lc_tile_44: READY
lc_tile_45: READY
lc_tile_46: READY
lc_tile_47: READY
lc_tile_48: CANCELLED
lc_tile_49: CANCELLED
lc_tile_50: CANCELLED
lc_tile_51: CANCELLED
lc_tile_52: CANCELLED
lc_tile_53: CANCELLED
lc_tile_54: CANCELLED
lc_tile_55: CANCELLED
lc_t

Streaming output truncated to the last 5000 lines.
lc_tile_5: READY
lc_tile_6: READY
lc_tile_7: READY
lc_tile_8: READY
lc_tile_9: READY
lc_tile_10: READY
lc_tile_11: READY
lc_tile_12: READY
lc_tile_13: READY
lc_tile_14: READY
lc_tile_15: READY
lc_tile_16: READY
lc_tile_17: READY
lc_tile_18: READY
lc_tile_19: READY
lc_tile_20: READY
lc_tile_21: READY
lc_tile_22: READY
lc_tile_23: READY
lc_tile_24: READY
lc_tile_25: READY
lc_tile_26: READY
lc_tile_27: READY
lc_tile_28: READY
lc_tile_29: READY
lc_tile_30: READY
lc_tile_31: READY
lc_tile_32: READY
lc_tile_33: READY
lc_tile_34: READY
lc_tile_35: READY
lc_tile_36: READY
lc_tile_37: READY
lc_tile_38: READY
lc_tile_39: READY
lc_tile_40: READY
lc_tile_41: READY
lc_tile_42: READY
lc_tile_43: READY
lc_tile_44: READY
lc_tile_45: READY
lc_tile_46: READY
lc_tile_47: READY
lc_tile_48: CANCELLED
lc_tile_49: CANCELLED
lc_tile_50: CANCELLED
lc_tile_51: CANCELLED
lc_tile_52: CANCELLED
lc_tile_53: CANCELLED
lc_tile_54: CANCELLED
lc_tile_55: CANCELLED
lc_t

lc_tile_0: READY
lc_tile_1: READY
lc_tile_2: READY
lc_tile_3: READY
lc_tile_4: READY
lc_tile_5: READY
lc_tile_6: READY
lc_tile_7: READY
lc_tile_8: READY
lc_tile_9: READY
lc_tile_10: READY
lc_tile_11: READY
lc_tile_12: READY
lc_tile_13: READY
lc_tile_14: READY
lc_tile_15: READY
lc_tile_16: READY
lc_tile_17: READY
lc_tile_18: READY
lc_tile_19: READY
lc_tile_20: READY
lc_tile_21: READY
lc_tile_22: READY
lc_tile_23: READY
lc_tile_24: READY
lc_tile_25: READY
lc_tile_26: READY
lc_tile_27: READY
lc_tile_28: READY
lc_tile_29: READY
lc_tile_30: READY
lc_tile_31: READY
lc_tile_32: READY
lc_tile_33: READY
lc_tile_34: READY
lc_tile_35: READY
lc_tile_36: READY
lc_tile_37: READY
lc_tile_38: READY
lc_tile_39: READY
lc_tile_40: READY
lc_tile_41: READY
lc_tile_42: READY
lc_tile_43: READY
lc_tile_44: READY
lc_tile_45: READY
lc_tile_46: READY
lc_tile_47: READY
lc_tile_48: CANCELLED
lc_tile_49: CANCELLED
lc_tile_50: CANCELLED
lc_tile_51: CANCELLED
lc_tile_52: CANCELLED
lc_tile_53: CANCELLED
lc_tile_54: CA

...

lc_tile_48: CANCELLED
lc_tile_49: CANCELLED
lc_tile_50: CANCELLED
lc_tile_51: CANCELLED
lc_tile_52: CANCELLED
lc_tile_53: CANCELLED
lc_tile_54: CANCELLED
lc_tile_55: CANCELLED
lc_tile_56: CANCELLED
lc_tile_57: CANCELLED
lc_tile_58: CANCELLED
lc_tile_59: CANCELLED
lc_tile_60: CANCELLED
lc_tile_61: CANCELLED
lc_tile_62: CANCELLED
Waiting 30 seconds...



lc_tile_0: FAILED
lc_tile_1: FAILED
lc_tile_2: FAILED
lc_tile_3: FAILED
lc_tile_4: FAILED
lc_tile_5: FAILED
lc_tile_6: COMPLETED
lc_tile_7: FAILED
lc_tile_8: FAILED
lc_tile_9: FAILED
lc_tile_10: FAILED
lc_tile_11: FAILED
lc_tile_12: COMPLETED
lc_tile_13: COMPLETED
lc_tile_14: FAILED
lc_tile_15: FAILED
lc_tile_16: FAILED
lc_tile_17: FAILED
lc_tile_18: COMPLETED
lc_tile_19: COMPLETED
lc_tile_20: COMPLETED
lc_tile_21: FAILED
lc_tile_22: FAILED
lc_tile_23: FAILED
lc_tile_24: RUNNING
lc_tile_25: READY
lc_tile_26: READY
lc_tile_27: READY
lc_tile_28: READY
lc_tile_29: READY
lc_tile_30: READY
lc_tile_31: READY
lc_tile_32: READY
lc_tile_33: READY
lc_tile_34: READY
lc_tile_35: READY
lc_tile_36: READY
lc_tile_37: READY
lc_tile_38: READY
lc_tile_39: READY
lc_tile_40: READY
lc_tile_41: READY
lc_tile_42: READY
lc_tile_43: READY
lc_tile_44: READY
lc_tile_45: READY
lc_tile_46: READY
lc_tile_47: READY
lc_tile_48: CANCELLED
lc_tile_49: CANCELLED
lc_tile_50: CANCELLED
lc_tile_51: CANCELLED
lc_tile_52: CA

lc_tile_26: READY
lc_tile_27: READY
lc_tile_28: READY
lc_tile_29: READY
lc_tile_30: READY
lc_tile_31: READY
lc_tile_32: READY
lc_tile_33: READY
lc_tile_34: READY
lc_tile_35: READY
lc_tile_36: READY
lc_tile_37: READY
lc_tile_38: READY
lc_tile_39: READY
lc_tile_40: READY
lc_tile_41: READY
lc_tile_42: READY
lc_tile_43: READY
lc_tile_44: READY
lc_tile_45: READY
lc_tile_46: READY
lc_tile_47: READY
lc_tile_48: CANCELLED
lc_tile_49: CANCELLED
lc_tile_50: CANCELLED
lc_tile_51: CANCELLED
lc_tile_52: CANCELLED
lc_tile_53: CANCELLED
lc_tile_54: CANCELLED
lc_tile_55: CANCELLED
lc_tile_56: CANCELLED
lc_tile_57: CANCELLED
lc_tile_58: CANCELLED
lc_tile_59: CANCELLED
lc_tile_60: CANCELLED
lc_tile_61: CANCELLED
lc_tile_62: CANCELLED
Waiting 30 seconds...

lc_tile_0: FAILED
lc_tile_1: FAILED
lc_tile_2: FAILED
lc_tile_3: FAILED
lc_tile_4: FAILED
lc_tile_5: FAILED
lc_tile_6: COMPLETED
lc_tile_7: FAILED
lc_tile_8: FAILED
lc_tile_9: FAILED
lc_tile_10: FAILED
lc_tile_11: FAILED
lc_tile_12: COMPLETED
lc_tile_

Streaming output truncated to the last 5000 lines.
lc_tile_5: FAILED
lc_tile_6: COMPLETED
lc_tile_7: FAILED
lc_tile_8: FAILED
lc_tile_9: FAILED
lc_tile_10: FAILED
lc_tile_11: FAILED
lc_tile_12: COMPLETED
lc_tile_13: COMPLETED
lc_tile_14: FAILED
lc_tile_15: FAILED
lc_tile_16: FAILED
lc_tile_17: FAILED
lc_tile_18: COMPLETED
lc_tile_19: COMPLETED
lc_tile_20: COMPLETED
lc_tile_21: FAILED
lc_tile_22: FAILED
lc_tile_23: FAILED
lc_tile_24: COMPLETED
lc_tile_25: COMPLETED
lc_tile_26: COMPLETED
lc_tile_27: COMPLETED
lc_tile_28: FAILED
lc_tile_29: FAILED
lc_tile_30: FAILED
lc_tile_31: COMPLETED
lc_tile_32: RUNNING
lc_tile_33: READY
lc_tile_34: READY
lc_tile_35: READY
lc_tile_36: READY
lc_tile_37: READY
lc_tile_38: READY
lc_tile_39: READY
lc_tile_40: READY
lc_tile_41: READY
lc_tile_42: READY
lc_tile_43: READY
lc_tile_44: READY
lc_tile_45: READY
lc_tile_46: READY
lc_tile_47: READY
lc_tile_48: CANCELLED
lc_tile_49: CANCELLED
lc_tile_50: CANCELLED
lc_tile_51: CANCELLED
lc_tile_52: CANCELLED
lc_tile_

lc_tile_0: FAILED
lc_tile_1: FAILED
lc_tile_2: FAILED
lc_tile_3: FAILED
lc_tile_4: FAILED
lc_tile_5: FAILED
lc_tile_6: COMPLETED
lc_tile_7: FAILED
lc_tile_8: FAILED
lc_tile_9: FAILED
lc_tile_10: FAILED
lc_tile_11: FAILED
lc_tile_12: COMPLETED
lc_tile_13: COMPLETED
lc_tile_14: FAILED
lc_tile_15: FAILED
lc_tile_16: FAILED
lc_tile_17: FAILED
lc_tile_18: COMPLETED
lc_tile_19: COMPLETED
lc_tile_20: COMPLETED
lc_tile_21: FAILED
lc_tile_22: FAILED
lc_tile_23: FAILED
lc_tile_24: COMPLETED
lc_tile_25: COMPLETED
lc_tile_26: COMPLETED
lc_tile_27: COMPLETED
lc_tile_28: FAILED
lc_tile_29: FAILED
lc_tile_30: FAILED
lc_tile_31: COMPLETED
lc_tile_32: COMPLETED
lc_tile_33: COMPLETED
lc_tile_34: COMPLETED
lc_tile_35: FAILED
lc_tile_36: FAILED
lc_tile_37: FAILED
lc_tile_38: FAILED
lc_tile_39: COMPLETED
lc_tile_40: COMPLETED
lc_tile_41: COMPLETED
lc_tile_42: FAILED
lc_tile_43: FAILED
lc_tile_44: FAILED
lc_tile_45: FAILED
lc_tile_46: FAILED
lc_tile_47: RUNNING
lc_tile_48: CANCELLED
lc_tile_49: CANCELLED
lc

In [ ]:
# for t in tasks:
#     status = t.status()
#     if status['state'] == 'FAILED':
#         print(f"{status['description']} failed: {status.get('error_message')}")